In [1]:

"""
DNA ENCODING + DNABRIDGE++ (novel transfer SSL) + FEDERATED (4 clients) + NESTED CV + FINAL TEST
- Dataset: loan_data.csv (Kaggle)  | Target: loan_status (auto-mapped to binary)
- No plots/images.
- Encoding validation tables computed ONCE for TrainVal (not per fold/client).

Fixes included:
✅ Fix sha256_row_fingerprint: avoids pandas .agg(...)->DataFrame .tolist() error
✅ Robust target mapping: auto-detects labels (Approved/Rejected, 1/0, Yes/No, True/False, etc.) so dataset won't become empty
✅ Robust numeric detection & parsing: handles commas/$/% and reduces false "categorical" numeric columns
✅ Safer ID-column dropping: won't drop continuous numeric columns just because they're near-unique

Deps:
  pip install pandas numpy scikit-learn torch

Run:
  python dna_dnabridgepp_fed_nestedcv_loanstatus.py

Optional:
  export LOAN_CSV="/full/path/to/loan_data.csv"
"""

import os
import math
import random
import hashlib
import warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    log_loss, roc_auc_score, average_precision_score
)
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# =========================
# 0) Silence known torch warning (nested tensor)
# =========================
warnings.filterwarnings(
    "ignore",
    message=r"enable_nested_tensor is True, but self\.use_nested_tensor is False.*",
    category=UserWarning,
)
warnings.filterwarnings("ignore", category=UserWarning, module=r"torch\.nn\.modules\.transformer")


# =========================
# 1) Repro + printing
# =========================
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def hline(char: str = "=", n: int = 120) -> str:
    return char * n

def _in_ipython() -> bool:
    try:
        from IPython import get_ipython  # noqa
        return get_ipython() is not None
    except Exception:
        return False

def print_table(df: pd.DataFrame, title: str, index: bool = False) -> None:
    print("\n" + hline("="))
    print(title)
    print(hline("="))

    if df is None or len(df) == 0:
        print("(empty)")
        return

    d = df.copy()
    for c in d.columns:
        if pd.api.types.is_float_dtype(d[c]):
            d[c] = d[c].astype(float)

    if _in_ipython():
        try:
            from IPython.display import display, Markdown  # noqa
            md = d.to_markdown(index=index, tablefmt="github", floatfmt=".6f")
            display(Markdown(md))
            return
        except Exception:
            pass

    with pd.option_context(
        "display.max_columns", None,
        "display.width", 5000,
        "display.expand_frame_repr", False,
        "display.max_colwidth", None,
    ):
        print(d.to_string(index=index, float_format=lambda x: f"{x:.6f}"))

def safe_nanmean(x: List[float]) -> float:
    x = np.array(x, dtype=np.float64)
    if np.all(np.isnan(x)):
        return float("nan")
    return float(np.nanmean(x))


# =========================
# 2) File auto-discovery
# =========================
def find_csv_path(preferred: str) -> str:
    if preferred and os.path.isfile(preferred):
        return preferred
    if preferred and os.path.isfile(os.path.join(os.getcwd(), preferred)):
        return os.path.join(os.getcwd(), preferred)

    search_roots = []
    if os.path.isdir("/kaggle/input"):
        search_roots.append("/kaggle/input")
    search_roots.append(os.getcwd())

    preferred_lower = (os.path.basename(preferred) if preferred else "").lower()
    best, best_score = None, -1

    for root in search_roots:
        for dirpath, _, filenames in os.walk(root):
            for fn in filenames:
                if not fn.lower().endswith(".csv"):
                    continue
                score = 0
                low = fn.lower()
                if preferred_lower and preferred_lower in low:
                    score += 10
                for kw in ["loan", "approval", "dataset", "predict"]:
                    if kw in low:
                        score += 2
                if score > best_score:
                    best_score = score
                    best = os.path.join(dirpath, fn)

    if best and os.path.isfile(best):
        return best

    raise FileNotFoundError(
        f"Could not find CSV.\nTried preferred='{preferred}'.\n"
        f"Set env:\n  export LOAN_CSV='/kaggle/input/.../loan_data.csv'\n"
        f"or place the CSV in the working directory."
    )


# =========================
# 3) Cleaning + robust target parsing
# =========================
def clean_numeric_series(s: pd.Series) -> pd.Series:
    """Robust numeric parsing for strings like '12,000', '$5000', '13.5%', ' 42 '."""
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce")
    ss = s.astype(str).fillna("NA").str.strip()
    ss = ss.str.replace(r"[,\$]", "", regex=True)
    ss = ss.str.replace("%", "", regex=False)
    ss = ss.str.replace(r"\s+", "", regex=True)
    return pd.to_numeric(ss, errors="coerce")

def sha256_row_fingerprint(df: pd.DataFrame) -> pd.Series:
    """
    Fixes: pandas DataFrame.agg(..., axis=1) may return a DataFrame in some versions -> no .tolist()
    """
    if len(df) == 0:
        return pd.Series([], dtype=str, index=df.index)

    arr = df.astype(str).fillna("NA").to_numpy(dtype=str)
    rows = ["||".join(r) for r in arr]
    dig = [hashlib.sha256(r.encode("utf-8")).hexdigest() for r in rows]
    return pd.Series(dig, index=df.index, dtype=str)

def drop_id_like_cols(df: pd.DataFrame, threshold_unique_ratio: float = 0.98) -> Tuple[pd.DataFrame, List[str]]:
    """
    Safer version: don't drop continuous numeric columns just because they are near-unique.
    """
    n = len(df)
    drop_cols: List[str] = []
    for c in df.columns:
        if n == 0:
            continue

        name = c.lower().strip()
        nunq = df[c].nunique(dropna=False)
        ratio = nunq / max(n, 1)

        conv = clean_numeric_series(df[c])
        numlike = float(conv.notna().mean()) >= 0.90

        is_id_name = ("id" in name) or name.endswith("_id") or name.startswith("id")

        if is_id_name and ratio >= 0.80:
            drop_cols.append(c)
        elif (not numlike) and ratio >= threshold_unique_ratio:
            drop_cols.append(c)

    drop_cols = sorted(list(set(drop_cols)))
    out = df.drop(columns=drop_cols) if drop_cols else df.copy()
    return out, drop_cols

def infer_numeric_vs_categorical(df: pd.DataFrame, target_col: str, numeric_threshold: float = 0.80) -> Tuple[List[str], List[str]]:
    num_cols, cat_cols = [], []
    for c in df.columns:
        if c == target_col:
            continue
        converted = clean_numeric_series(df[c])
        frac = float(converted.notna().mean())
        if frac >= numeric_threshold:
            num_cols.append(c)
        else:
            cat_cols.append(c)
    return num_cols, cat_cols

def infer_binary_target(y_raw: pd.Series) -> Tuple[np.ndarray, np.ndarray, Dict[str, Any]]:
    """
    Auto-map any 2-class target to {0,1} without hardcoding "Approved/Rejected".

    Returns:
      y (np.int64): 0/1 labels
      keep_mask (np.bool_): which rows were kept (drops rare/unknown classes if present)
      info dict: mapping details
    """
    s = y_raw.copy()

    # numeric mode (e.g., 0/1)
    y_num = clean_numeric_series(s)
    if float(y_num.notna().mean()) >= 0.90:
        vals = y_num.dropna().astype(int)
        vc = vals.value_counts()
        top2 = vc.index.tolist()[:2]
        if len(top2) < 2:
            raise ValueError(f"Target has <2 classes after numeric parsing. value_counts={vc.to_dict()}")
        neg_val, pos_val = int(min(top2)), int(max(top2))
        keep = y_num.isin([neg_val, pos_val])
        y = (y_num[keep].astype(int) == pos_val).astype(np.int64).to_numpy()
        info = {
            "mode": "numeric",
            "selected_classes": [neg_val, pos_val],
            "positive_class": pos_val,
            "negative_class": neg_val,
            "kept_rows": int(keep.sum()),
            "dropped_rows": int((~keep).sum()),
            "raw_unique_preview": y_raw.astype(str).value_counts().head(10).to_dict(),
        }
        return y, keep.to_numpy(), info

    # string mode
    s_str = s.astype(str).fillna("NA").str.strip()
    s_norm = s_str.str.lower()

    pos_set = {"approved", "approve", "accept", "accepted", "yes", "y", "true", "t", "1", "pos", "positive"}
    neg_set = {"rejected", "reject", "deny", "denied", "no", "n", "false", "f", "0", "neg", "negative"}

    vc = s_norm.value_counts()
    top2 = vc.index.tolist()[:2]
    if len(top2) < 2:
        raise ValueError(f"Target has <2 classes. value_counts={vc.to_dict()}")

    a, b = top2[0], top2[1]

    def score_label(x: str) -> int:
        if x in pos_set:
            return 2
        if x in neg_set:
            return 0
        try:
            xi = int(float(x))
            return 2 if xi == 1 else 0 if xi == 0 else 1
        except Exception:
            return 1

    sa, sb = score_label(a), score_label(b)
    if sa == sb:
        # deterministic fallback
        pos, neg = (b, a) if b > a else (a, b)
    else:
        pos = a if sa > sb else b
        neg = b if pos == a else a

    keep = s_norm.isin([pos, neg])
    y = (s_norm[keep] == pos).astype(np.int64).to_numpy()
    info = {
        "mode": "string",
        "selected_classes": [neg, pos],
        "positive_class": pos,
        "negative_class": neg,
        "kept_rows": int(keep.sum()),
        "dropped_rows": int((~keep).sum()),
        "raw_unique_preview": y_raw.astype(str).value_counts().head(10).to_dict(),
    }
    return y, keep.to_numpy(), info


# =========================
# 4) DNA helpers
# =========================
DNA_ALPHABET = np.array(list("ACGT"))

def int_to_kmer(i: int, k: int) -> str:
    digits = []
    x = int(i)
    for _ in range(k):
        digits.append(int(x % 4))
        x //= 4
    digits = digits[::-1]
    return "".join(DNA_ALPHABET[digits].tolist())


# =========================
# 5) Improved encoding: supervised numeric edges + collision-free categorical vocab
# =========================
def quantile_edges(x: np.ndarray, bins: int) -> np.ndarray:
    x = x.astype(np.float64)
    if bins <= 1 or len(x) == 0:
        return np.array([], dtype=np.float64)
    qs = np.linspace(0, 1, bins + 1)[1:-1]
    edges = np.quantile(x, qs)
    edges = np.unique(edges)
    return edges.astype(np.float64)

def supervised_quantile_edges(x: np.ndarray, y: np.ndarray, bins: int, seed: int = 42) -> np.ndarray:
    x = x.astype(np.float64)
    y = y.astype(np.int64)
    if bins <= 1 or len(x) == 0:
        return np.array([], dtype=np.float64)

    b_global = max(4, bins // 2)
    b_class = max(3, bins // 4)

    e_all = quantile_edges(x, b_global)
    e0 = quantile_edges(x[y == 0], b_class) if np.any(y == 0) else np.array([], dtype=np.float64)
    e1 = quantile_edges(x[y == 1], b_class) if np.any(y == 1) else np.array([], dtype=np.float64)

    parts = [p for p in [e_all, e0, e1] if len(p)]
    edges = np.unique(np.concatenate(parts, axis=0)) if parts else np.array([], dtype=np.float64)
    edges = np.sort(edges)

    max_edges = bins - 1
    if len(edges) > max_edges:
        idx = np.linspace(0, len(edges) - 1, max_edges).round().astype(int)
        edges = np.unique(edges[idx])

    return edges.astype(np.float64)

class CategoricalVocabEncoder:
    """
    Deterministic, collision-free per-column vocab:
      - OOV token id = 0
      - top-(max_vocab-1) categories get ids 1..K
    """
    def __init__(self, max_vocab: int = 256):
        self.max_vocab = int(max_vocab)
        self.maps: Dict[str, Dict[str, int]] = {}

    def fit(self, dfX: pd.DataFrame, cat_cols: List[str]) -> None:
        self.maps = {}
        for c in cat_cols:
            s = dfX[c].astype(str).fillna("NA")
            vc = s.value_counts(dropna=False)
            keep = vc.index.tolist()[: max(1, self.max_vocab - 1)]
            self.maps[c] = {k: (i + 1) for i, k in enumerate(keep)}  # 1..K

    def transform_col(self, s: pd.Series, col: str) -> np.ndarray:
        mp = self.maps.get(col, {})
        v = s.astype(str).fillna("NA").map(mp)
        v = v.fillna(0).astype(np.int64).to_numpy()
        v = np.clip(v, 0, self.max_vocab - 1)
        return v

    def transform(self, dfX: pd.DataFrame, cat_cols: List[str]) -> Dict[str, np.ndarray]:
        return {c: self.transform_col(dfX[c], c) for c in cat_cols}

@dataclass
class FittedDNAEncoder:
    target_col: str
    num_cols: List[str]
    cat_cols: List[str]
    bins_numeric: int
    vocab_cat: int
    num_impute_mean: Dict[str, float]
    num_mu: Dict[str, float]
    num_sd: Dict[str, float]
    edges_by_col: Dict[str, np.ndarray]
    cat_encoder: CategoricalVocabEncoder
    feat_names: List[str]
    cardinalities: List[int]
    k_num: int
    k_cat: int

def fit_dna_encoder(
    df_train: pd.DataFrame,
    y_train: np.ndarray,
    target_col: str,
    num_cols: List[str],
    cat_cols: List[str],
    bins_numeric: int = 128,
    vocab_cat: int = 256,
    seed: int = 42,
) -> FittedDNAEncoder:
    set_seed(seed)
    dfX = df_train.copy()

    num_impute_mean: Dict[str, float] = {}
    for c in num_cols:
        x = clean_numeric_series(dfX[c])
        mu = float(x.mean()) if x.notna().any() else 0.0
        num_impute_mean[c] = mu
        dfX[c] = x.fillna(mu)

    num_mu, num_sd = {}, {}
    for c in num_cols:
        x = dfX[c].to_numpy(dtype=np.float64)
        num_mu[c] = float(np.mean(x)) if len(x) else 0.0
        num_sd[c] = float(np.std(x) + 1e-12) if len(x) else 1.0

    edges_by_col = {}
    for c in num_cols:
        x = dfX[c].to_numpy(dtype=np.float64)
        edges_by_col[c] = supervised_quantile_edges(x, y_train, bins=bins_numeric, seed=seed)

    cat_enc = CategoricalVocabEncoder(max_vocab=vocab_cat)
    cat_enc.fit(dfX, cat_cols)

    feat_names = [*num_cols, *cat_cols]
    cardinalities = [bins_numeric] * len(num_cols) + [vocab_cat] * len(cat_cols)

    k_num = int(math.ceil(math.log(max(bins_numeric, 2), 4)))
    k_cat = int(math.ceil(math.log(max(vocab_cat, 2), 4)))

    return FittedDNAEncoder(
        target_col=target_col,
        num_cols=num_cols,
        cat_cols=cat_cols,
        bins_numeric=bins_numeric,
        vocab_cat=vocab_cat,
        num_impute_mean=num_impute_mean,
        num_mu=num_mu,
        num_sd=num_sd,
        edges_by_col=edges_by_col,
        cat_encoder=cat_enc,
        feat_names=feat_names,
        cardinalities=cardinalities,
        k_num=k_num,
        k_cat=k_cat,
    )

def transform_dna(enc: FittedDNAEncoder, df_any: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    dfX = df_any.copy()

    numeric_std_parts: List[np.ndarray] = []
    num_token_parts: List[np.ndarray] = []

    for c in enc.num_cols:
        x = clean_numeric_series(dfX[c]).fillna(enc.num_impute_mean[c]).to_numpy(dtype=np.float64)
        mu, sd = enc.num_mu[c], enc.num_sd[c]
        numeric_std_parts.append(((x - mu) / sd).astype(np.float32))

        edges = enc.edges_by_col[c]
        t = np.digitize(x, edges, right=False).astype(np.int64)
        t = np.clip(t, 0, enc.bins_numeric - 1)
        num_token_parts.append(t.reshape(-1, 1))

    cat_tokens = enc.cat_encoder.transform(dfX, enc.cat_cols)
    cat_token_parts = [cat_tokens[c].reshape(-1, 1) for c in enc.cat_cols]

    parts = [*num_token_parts, *cat_token_parts]
    X_local = np.concatenate(parts, axis=1).astype(np.int64) if parts else np.zeros((len(dfX), 0), dtype=np.int64)

    numeric_std = (
        np.stack(numeric_std_parts, axis=1).astype(np.float32)
        if numeric_std_parts else np.zeros((len(dfX), 0), dtype=np.float32)
    )
    return X_local, numeric_std


# =========================
# 6) Encoding validation (computed ONCE)
# =========================
def _entropy_from_counts(counts: np.ndarray) -> float:
    p = counts.astype(np.float64)
    p = p / (p.sum() + 1e-12)
    p = np.clip(p, 1e-12, 1.0)
    return float(-np.sum(p * np.log(p)))

def validate_numeric_dna_quantile(
    dfX: pd.DataFrame,
    y: np.ndarray,
    num_cols: List[str],
    edges_by_col: Dict[str, np.ndarray],
    bins_numeric: int,
    silhouette_subset: int = 2000,
    seed: int = 42,
) -> pd.DataFrame:
    if len(num_cols) == 0:
        return pd.DataFrame([])

    y_counts = np.bincount(y, minlength=2)
    Hy = _entropy_from_counts(y_counts)

    rng = np.random.default_rng(seed)
    idx_sub = rng.choice(np.arange(len(dfX)), size=min(silhouette_subset, len(dfX)), replace=False)

    rows = []
    for c in num_cols:
        x = clean_numeric_series(dfX[c]).fillna(clean_numeric_series(dfX[c]).mean()).to_numpy(dtype=np.float64)
        edges = edges_by_col[c]
        t = np.clip(np.digitize(x, edges, right=False).astype(np.int64), 0, bins_numeric - 1)

        counts = np.bincount(t, minlength=bins_numeric)
        Ht = _entropy_from_counts(counts)
        eff = float(np.exp(Ht))
        uniq_used = int(np.sum(counts > 0))

        xs = x[idx_sub].reshape(-1, 1)
        ts = t[idx_sub]
        k = int(len(np.unique(ts)))

        sil = float("nan")
        dbi = float("nan")
        chi = float("nan")
        if k >= 2 and len(xs) >= 10:
            try:
                sil = float(silhouette_score(xs, ts, metric="euclidean"))
            except Exception:
                pass
            try:
                dbi = float(davies_bouldin_score(xs, ts))
            except Exception:
                pass
            try:
                chi = float(calinski_harabasz_score(xs, ts))
            except Exception:
                pass

        cent = np.zeros(bins_numeric, dtype=np.float64)
        x_mean = float(np.mean(x)) if len(x) else 0.0
        for b in range(bins_numeric):
            mask = (t == b)
            cent[b] = float(np.mean(x[mask])) if np.any(mask) else x_mean
        xhat = cent[t]
        q_mse = float(np.mean((x - xhat) ** 2)) if len(x) else float("nan")

        mi = float("nan")
        try:
            mi = float(mutual_info_classif(t.reshape(-1, 1), y, discrete_features=True, random_state=seed)[0])
        except Exception:
            pass
        nmi = float(mi / Hy) if (Hy > 1e-12 and mi == mi) else float("nan")

        rows.append({
            "feature": c,
            "bins_vocab_size": bins_numeric,
            "edges_count": int(len(edges)),
            "edges_preview_first10": str(edges[:10].tolist()) + ("..." if len(edges) > 10 else ""),
            "unique_tokens_used": int(np.sum(counts > 0)),
            "silhouette": sil,
            "davies_bouldin": dbi,
            "calinski_harabasz": chi,
            "quantization_mse": q_mse,
            "token_entropy": Ht,
            "effective_tokens_expH": eff,
            "mutual_info_with_target": mi,
            "normalized_mi": nmi,
        })

    return pd.DataFrame(rows)

def validate_categorical_vocab(
    dfX: pd.DataFrame,
    cat_cols: List[str],
    cat_encoder: CategoricalVocabEncoder,
    vocab_cat: int,
) -> pd.DataFrame:
    if len(cat_cols) == 0:
        return pd.DataFrame([])

    rows = []
    for c in cat_cols:
        raw = dfX[c].astype(str).fillna("NA")
        tok = cat_encoder.transform_col(raw, c)
        n = len(tok)

        unique_raw = int(raw.nunique(dropna=False))
        occupied = int(len(np.unique(tok)))

        oov_rate = float(np.mean(tok == 0))
        counts = np.bincount(tok, minlength=vocab_cat).astype(np.float64)
        nonzero = counts[counts > 0]
        Hb = _entropy_from_counts(nonzero) if len(nonzero) else float("nan")
        eff = float(np.exp(Hb)) if Hb == Hb else float("nan")

        p = nonzero / (nonzero.sum() + 1e-12) if len(nonzero) else np.array([])
        top_share = float(np.max(p)) if len(p) else float("nan")

        rows.append({
            "feature": c,
            "vocab_size": vocab_cat,
            "n_samples": n,
            "unique_raw_categories": unique_raw,
            "occupied_tokens": occupied,
            "oov_rate": oov_rate,
            "token_entropy": Hb,
            "effective_tokens_expH": eff,
            "top_token_share": top_share,
            "mean_token_load": float(n / max(occupied, 1)),
            "max_token_load": int(np.max(nonzero)) if len(nonzero) else 0,
        })

    return pd.DataFrame(rows)

def edge_deviation(base: np.ndarray, boot: np.ndarray, x_scale: float) -> float:
    b = np.array(sorted(base.tolist()), dtype=np.float64)
    t = np.array(sorted(boot.tolist()), dtype=np.float64)
    if len(b) == 0 and len(t) == 0:
        return 0.0
    if len(b) == 0 or len(t) == 0:
        return float(x_scale)
    m = min(len(b), len(t))
    dev = float(np.mean(np.abs(b[:m] - t[:m])))
    if len(b) != len(t):
        dev += float(abs(len(b) - len(t)) * 0.25 * x_scale)
    return dev

def bootstrap_stability_numeric_edges(
    dfX: pd.DataFrame,
    y: np.ndarray,
    num_cols: List[str],
    edges_by_col: Dict[str, np.ndarray],
    bins_numeric: int,
    B: int = 6,
    sample_frac: float = 0.7,
    fixed_compare_n: int = 2000,
    seed: int = 42,
) -> pd.DataFrame:
    if len(num_cols) == 0:
        return pd.DataFrame([])

    rng = np.random.default_rng(seed)
    n = len(dfX)
    fixed_idx = np.arange(min(n, fixed_compare_n))

    rows = []
    for c in num_cols:
        x_full = clean_numeric_series(dfX[c]).fillna(clean_numeric_series(dfX[c]).mean()).to_numpy(dtype=np.float64)
        base_edges = edges_by_col[c]
        x_scale = float(np.std(x_full) + 1e-12)

        base_t_fixed = np.clip(np.digitize(x_full[fixed_idx], base_edges, right=False), 0, bins_numeric - 1)

        devs, agrees, bins_used = [], [], []
        for _ in range(B):
            idx = rng.choice(np.arange(n), size=int(sample_frac * n), replace=True)
            edges_bs = supervised_quantile_edges(x_full[idx], y[idx], bins=bins_numeric, seed=seed)

            devs.append(edge_deviation(base_edges, edges_bs, x_scale=x_scale))

            t_bs_fixed = np.clip(np.digitize(x_full[fixed_idx], edges_bs, right=False), 0, bins_numeric - 1)
            agrees.append(float(np.mean(t_bs_fixed == base_t_fixed)))
            bins_used.append(int(len(np.unique(t_bs_fixed))))

        rows.append({
            "feature": c,
            "bins_vocab_size": bins_numeric,
            "base_edges_count": int(len(base_edges)),
            "base_std_scale": x_scale,
            "bootstrap_B": B,
            "mean_abs_edge_deviation": float(np.mean(devs)),
            "std_abs_edge_deviation": float(np.std(devs)),
            "mean_token_agreement_vs_base": float(np.mean(agrees)),
            "std_token_agreement_vs_base": float(np.std(agrees)),
            "mean_bins_used_fixed_subset": float(np.mean(bins_used)),
        })

    return pd.DataFrame(rows)

def token_robustness_under_mutation(
    dfX: pd.DataFrame,
    num_cols: List[str],
    cat_cols: List[str],
    edges_by_col: Dict[str, np.ndarray],
    cat_encoder: CategoricalVocabEncoder,
    bins_numeric: int,
    mutation_rate: float = 0.10,
    seed: int = 42,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    rows = []

    for j, c in enumerate(num_cols):
        x = clean_numeric_series(dfX[c]).fillna(clean_numeric_series(dfX[c]).mean()).to_numpy(dtype=np.float64)
        edges = edges_by_col[c]
        t0 = np.clip(np.digitize(x, edges, right=False), 0, bins_numeric - 1)

        sigma = mutation_rate * (np.std(x) + 1e-12)
        x_mut = x + rng.normal(0.0, sigma, size=len(x))
        t1 = np.clip(np.digitize(x_mut, edges, right=False), 0, bins_numeric - 1)

        rows.append({
            "feature_index": j,
            "feature": c,
            "cardinality": bins_numeric,
            "is_numeric": True,
            "mutation_rate": mutation_rate,
            "token_agreement": float(np.mean(t1 == t0)),
        })

    for j, c in enumerate(cat_cols):
        raw = dfX[c].astype(str).fillna("NA").to_numpy()
        t0 = cat_encoder.transform_col(pd.Series(raw), c)

        raw_mut = raw.copy()
        m = rng.random(len(raw_mut)) < mutation_rate
        rand_chars = rng.choice(list("ACGT"), size=np.sum(m))
        raw_mut[m] = np.char.add(raw_mut[m].astype(str), rand_chars.astype(str))
        t1 = cat_encoder.transform_col(pd.Series(raw_mut), c)

        rows.append({
            "feature_index": len(num_cols) + j,
            "feature": c,
            "cardinality": cat_encoder.max_vocab,
            "is_numeric": False,
            "mutation_rate": mutation_rate,
            "token_agreement": float(np.mean(t1 == t0)),
        })

    return pd.DataFrame(rows)


# =========================
# 7) Token offsets + augmentation
# =========================
def build_offsets(cardinalities: List[int]) -> Tuple[List[int], int]:
    offsets = []
    cur = 0
    for card in cardinalities:
        offsets.append(cur)
        cur += int(card)
    return offsets, cur  # token_base

def to_global_token_ids(local_tokens: torch.Tensor, offsets: torch.Tensor, token_base: int) -> torch.Tensor:
    B, F_ = local_tokens.shape
    glob = torch.empty_like(local_tokens, dtype=torch.long)
    for j in range(F_):
        v = local_tokens[:, j]
        mask = (v == -1)
        glob[~mask, j] = offsets[j] + v[~mask].long()
        glob[mask, j] = token_base + j
    return glob

def make_augmented_view_local(
    X_local: np.ndarray,
    cardinalities: List[int],
    num_feature_count: int,
    p_mask: float = 0.22,
    p_jitter_numeric: float = 0.18,
    p_jitter_cat: float = 0.08,
    seed: int = 42,
) -> np.ndarray:
    rng = np.random.default_rng(seed)
    X = X_local.copy()
    N, F_ = X.shape

    mask = rng.random((N, F_)) < p_mask

    for j in range(F_):
        card = int(cardinalities[j])
        if card <= 1:
            continue
        if j < num_feature_count:
            jit = rng.random(N) < p_jitter_numeric
            idx = np.where((~mask[:, j]) & jit)[0]
            if len(idx) > 0:
                delta = rng.choice([-1, 1], size=len(idx))
                X[idx, j] = np.clip(X[idx, j] + delta, 0, card - 1)
        else:
            jit = rng.random(N) < p_jitter_cat
            idx = np.where((~mask[:, j]) & jit)[0]
            if len(idx) > 0:
                X[idx, j] = rng.integers(0, card, size=len(idx))

    X[mask] = -1
    return X.astype(np.int64)


# =========================
# 8) DNABRIDGE++ (SSL encoder + masked recon + VICReg)
# =========================
def vicreg_loss(z1: torch.Tensor, z2: torch.Tensor, sim_coeff=25.0, std_coeff=25.0, cov_coeff=1.0) -> torch.Tensor:
    sim = F.mse_loss(z1, z2)

    eps = 1e-4
    std_z1 = torch.sqrt(z1.var(dim=0) + eps)
    std_z2 = torch.sqrt(z2.var(dim=0) + eps)
    std = torch.mean(F.relu(1.0 - std_z1)) + torch.mean(F.relu(1.0 - std_z2))

    z1 = z1 - z1.mean(dim=0)
    z2 = z2 - z2.mean(dim=0)
    N, D = z1.shape
    cov_z1 = (z1.T @ z1) / (N - 1 + 1e-12)
    cov_z2 = (z2.T @ z2) / (N - 1 + 1e-12)

    def offdiag(m: torch.Tensor) -> torch.Tensor:
        return m.flatten()[:-1].view(D - 1, D + 1)[:, 1:].flatten()

    cov = (offdiag(cov_z1).pow(2).mean() + offdiag(cov_z2).pow(2).mean())
    return sim_coeff * sim + std_coeff * std + cov_coeff * cov

class DNABridgePPEncoder(nn.Module):
    def __init__(
        self,
        cardinalities: List[int],
        num_feature_count: int,
        emb_dim: int = 96,
        n_layers: int = 2,
        n_heads: int = 4,
        ff_dim: int = 192,
        dropout: float = 0.10,
    ):
        super().__init__()
        self.cardinalities = list(cardinalities)
        self.num_feature_count = int(num_feature_count)
        self.F_ = len(cardinalities)

        offsets, token_base = build_offsets(cardinalities)
        self.offsets_list = offsets
        self.token_base = int(token_base)

        self.total_tokens = self.token_base + self.F_ + 1  # +mask tokens + CLS
        self.cls_id = self.token_base + self.F_

        self.emb = nn.Embedding(self.total_tokens, emb_dim)

        self.pos_emb = nn.Parameter(torch.zeros(self.F_ + 1, emb_dim))
        nn.init.normal_(self.pos_emb, mean=0.0, std=0.02)

        self.num_pos_emb = nn.Parameter(torch.zeros(self.F_ + 1, emb_dim))
        nn.init.normal_(self.num_pos_emb, mean=0.0, std=0.02)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim, nhead=n_heads, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True, activation="gelu",
            norm_first=False,  # ✅ avoids nested-tensor warning
        )
        self.tr = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

        self.proj = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, emb_dim),
        )

        self.register_buffer("offsets", torch.tensor(self.offsets_list, dtype=torch.long), persistent=False)
        self.numeric_std_dim = self.num_feature_count

    def forward(self, local_tokens: torch.Tensor, numeric_std: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        B, F_ = local_tokens.shape
        assert F_ == self.F_

        glob = to_global_token_ids(local_tokens, self.offsets, self.token_base)
        cls = torch.full((B, 1), fill_value=self.cls_id, dtype=torch.long, device=glob.device)
        glob = torch.cat([cls, glob], dim=1)

        x = self.emb(glob)

        num_full = torch.zeros((B, self.F_ + 1), dtype=torch.float32, device=x.device)
        if self.numeric_std_dim > 0 and numeric_std.numel() > 0:
            for k in range(self.numeric_std_dim):
                num_full[:, 1 + k] = numeric_std[:, k]

        x = x + self.pos_emb.unsqueeze(0) + (num_full.unsqueeze(-1) * self.num_pos_emb.unsqueeze(0))
        out = self.tr(x)

        cls_out = out[:, 0, :]
        tok_out = out[:, 1:, :]
        z = self.proj(cls_out)
        return cls_out, tok_out, F.normalize(z, dim=1)

    def count_params(self) -> int:
        return int(sum(p.numel() for p in self.parameters()))

class DNABridgePP_SSL(nn.Module):
    def __init__(self, encoder: DNABridgePPEncoder):
        super().__init__()
        self.encoder = encoder
        self.token_base = encoder.token_base
        D = encoder.emb.embedding_dim
        self.recon = nn.Linear(D, self.token_base)

    def forward_ssl(
        self,
        view_local: torch.Tensor,
        numeric_std: torch.Tensor,
        target_global: torch.Tensor,
        mask_positions: torch.Tensor,
    ) -> torch.Tensor:
        _, tok_out, _ = self.encoder(view_local, numeric_std)
        logits = self.recon(tok_out)

        if mask_positions.sum().item() == 0:
            return torch.tensor(0.0, device=logits.device)

        logits_m = logits[mask_positions]
        target_m = target_global[mask_positions]
        return F.cross_entropy(logits_m, target_m.long())


# =========================
# 9) Federated SSL helpers
# =========================
@dataclass
class SSLCombo:
    lr: float
    local_epochs: int

class TokenOnlyDataset(Dataset):
    def __init__(self, X_local: np.ndarray, numeric_std: np.ndarray):
        self.X = X_local.astype(np.int64)
        self.N = numeric_std.astype(np.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx: int):
        return self.X[idx], self.N[idx]

def run_ssl_epoch(
    model_ssl: DNABridgePP_SSL,
    loader: DataLoader,
    cardinalities: List[int],
    num_feature_count: int,
    device: str,
    opt: Optional[torch.optim.Optimizer],
    train: bool,
    p_mask: float,
    seed: int,
    mlm_coeff: float = 1.0,
    vicreg_coeff: float = 0.25,
) -> float:
    model_ssl.train() if train else model_ssl.eval()

    losses = []
    step = 0
    for Xb, Nb in loader:
        Xb = Xb.to(device)
        Nb = Nb.to(device)

        v1 = make_augmented_view_local(
            Xb.cpu().numpy(), cardinalities, num_feature_count,
            p_mask=p_mask, seed=seed + step * 17 + 1
        )
        v2 = make_augmented_view_local(
            Xb.cpu().numpy(), cardinalities, num_feature_count,
            p_mask=p_mask, seed=seed + step * 17 + 2
        )

        v1t = torch.from_numpy(v1).to(device)
        v2t = torch.from_numpy(v2).to(device)

        target_global = to_global_token_ids(Xb, model_ssl.encoder.offsets, model_ssl.encoder.token_base)
        mask_pos = (v1t == -1)

        with torch.set_grad_enabled(train):
            loss_mlm = model_ssl.forward_ssl(v1t, Nb, target_global, mask_pos)
            _, _, z1 = model_ssl.encoder(v1t, Nb)
            _, _, z2 = model_ssl.encoder(v2t, Nb)
            loss_v = vicreg_loss(z1, z2)

            loss = mlm_coeff * loss_mlm + vicreg_coeff * loss_v

            if train and opt is not None:
                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()

        losses.append(float(loss.detach().cpu()))
        step += 1

    return float(np.mean(losses)) if losses else float("nan")

def fedavg(state_dicts: List[Dict[str, torch.Tensor]], weights: List[int]) -> Dict[str, torch.Tensor]:
    total = float(sum(weights))
    out: Dict[str, torch.Tensor] = {}
    keys = state_dicts[0].keys()
    for k in keys:
        acc = None
        for sd, w in zip(state_dicts, weights):
            t = sd[k].detach().cpu() * (w / total)
            acc = t if acc is None else (acc + t)
        out[k] = acc
    return out


# =========================
# 10) Supervised fine-tune + metrics (opt threshold for ACC)
# =========================
@dataclass
class SupCombo:
    lr: float
    freeze_encoder: bool

class SupDataset(Dataset):
    def __init__(self, X_local: np.ndarray, numeric_std: np.ndarray, y: np.ndarray):
        self.X = X_local.astype(np.int64)
        self.N = numeric_std.astype(np.float32)
        self.y = y.astype(np.int64)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx: int):
        return self.X[idx], self.N[idx], self.y[idx]

class DNABridgePP_Classifier(nn.Module):
    def __init__(self, encoder: DNABridgePPEncoder, numeric_dim: int, head_hidden: int = 192, dropout: float = 0.10):
        super().__init__()
        self.encoder = encoder
        D = encoder.emb.embedding_dim

        self.numeric_mlp = nn.Sequential(
            nn.Linear(numeric_dim, D),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(D, D),
        ) if numeric_dim > 0 else None

        self.head = nn.Sequential(
            nn.Linear(D + (D if numeric_dim > 0 else 0), head_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden, 1),
        )

    def forward(self, X_local: torch.Tensor, numeric_std: torch.Tensor) -> torch.Tensor:
        cls_out, _, _ = self.encoder(X_local, numeric_std)
        if self.numeric_mlp is not None and numeric_std.numel() > 0:
            nemb = self.numeric_mlp(numeric_std)
            z = torch.cat([cls_out, nemb], dim=1)
        else:
            z = cls_out
        return self.head(z).squeeze(1)

def compute_metrics(y_true: np.ndarray, prob_pos: np.ndarray, thr: float = 0.5) -> Dict[str, float]:
    y_true = y_true.astype(int)
    prob_pos = prob_pos.astype(float)
    y_pred = (prob_pos >= thr).astype(int)

    out: Dict[str, float] = {}
    out["acc"] = float(accuracy_score(y_true, y_pred))
    out["precision"] = float(precision_score(y_true, y_pred, zero_division=0))
    out["recall"] = float(recall_score(y_true, y_pred, zero_division=0))
    out["f1"] = float(f1_score(y_true, y_pred, zero_division=0))
    out["logloss"] = float(log_loss(y_true, np.clip(prob_pos, 1e-9, 1 - 1e-9), labels=[0, 1]))

    try:
        out["roc_auc"] = float(roc_auc_score(y_true, prob_pos))
    except Exception:
        out["roc_auc"] = float("nan")
    try:
        out["pr_auc"] = float(average_precision_score(y_true, prob_pos))
    except Exception:
        out["pr_auc"] = float("nan")

    out["auc"] = out["roc_auc"]
    return out

def pick_best_threshold(y_true: np.ndarray, prob_pos: np.ndarray, mode: str = "acc") -> Tuple[float, Dict[str, float]]:
    best_thr = 0.5
    best_score = None
    best_m = None

    for thr in np.linspace(0.05, 0.95, 91):
        m = compute_metrics(y_true, prob_pos, thr=float(thr))
        score = (m["acc"], m["f1"]) if mode == "acc" else (m["f1"], m["acc"])
        if best_score is None or score > best_score:
            best_score = score
            best_thr = float(thr)
            best_m = m

    return best_thr, (best_m if best_m is not None else compute_metrics(y_true, prob_pos, thr=0.5))

def eval_model_probs(model: nn.Module, loader: DataLoader, device: str) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for Xb, Nb, yb in loader:
            Xb = Xb.to(device)
            Nb = Nb.to(device)
            logits = model(Xb, Nb)
            prob = torch.sigmoid(logits).detach().cpu().numpy()
            ys.append(yb.numpy())
            ps.append(prob)
    return np.concatenate(ys), np.concatenate(ps)

def train_supervised(
    encoder_state: Dict[str, torch.Tensor],
    cardinalities: List[int],
    num_feature_count: int,
    numeric_dim: int,
    X_tr: np.ndarray,
    N_tr: np.ndarray,
    y_tr: np.ndarray,
    X_va: np.ndarray,
    N_va: np.ndarray,
    y_va: np.ndarray,
    lr: float,
    freeze_encoder: bool,
    device: str,
    max_epochs: int = 12,
    patience: int = 3,
    batch_size: int = 256,
    weight_decay: float = 1e-4,
    seed: int = 42,
    prefix: str = "[SUP]",
    thr_opt: str = "acc",
) -> Tuple[nn.Module, float, float]:
    set_seed(seed)

    enc = DNABridgePPEncoder(
        cardinalities=cardinalities,
        num_feature_count=num_feature_count,
        emb_dim=96, n_layers=2, n_heads=4, ff_dim=192, dropout=0.10
    ).to(device)
    enc.load_state_dict(encoder_state, strict=True)

    model = DNABridgePP_Classifier(enc, numeric_dim=numeric_dim, head_hidden=224, dropout=0.10).to(device)

    if freeze_encoder:
        for p in model.encoder.parameters():
            p.requires_grad = False

    pos = float(np.sum(y_tr == 1))
    neg = float(np.sum(y_tr == 0))
    pos_weight = torch.tensor([neg / (pos + 1e-12)], dtype=torch.float32, device=device)

    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=weight_decay)

    tr_loader = DataLoader(SupDataset(X_tr, N_tr, y_tr), batch_size=batch_size, shuffle=True, drop_last=False)
    va_loader = DataLoader(SupDataset(X_va, N_va, y_va), batch_size=batch_size, shuffle=False, drop_last=False)

    best_auc = -1.0
    best_state = None
    bad = 0

    for ep in range(1, max_epochs + 1):
        model.train()
        tr_losses = []
        for Xb, Nb, yb in tr_loader:
            Xb = Xb.to(device)
            Nb = Nb.to(device)
            yb = yb.to(device).float()
            logits = model(Xb, Nb)
            loss = crit(logits, yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            tr_losses.append(float(loss.detach().cpu()))

        yv, pv = eval_model_probs(model, va_loader, device=device)
        try:
            val_auc = float(roc_auc_score(yv, pv))
        except Exception:
            val_auc = float("nan")

        tr_loss = float(np.mean(tr_losses)) if tr_losses else float("nan")
        print(f"    {prefix} epoch={ep:02d} | train_loss={tr_loss:.4f} | val_roc_auc={val_auc:.4f}")

        if val_auc == val_auc and val_auc > best_auc + 1e-6:
            best_auc = val_auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state, strict=True)

    yv, pv = eval_model_probs(model, va_loader, device=device)
    best_thr, _ = pick_best_threshold(yv, pv, mode=thr_opt)
    return model, float(best_auc), float(best_thr)


# =========================
# 11) Nested CV selection + evaluation
# =========================
def inner_cv_select_combo(
    encoder_state: Dict[str, torch.Tensor],
    cardinalities: List[int],
    num_feature_count: int,
    numeric_dim: int,
    X: np.ndarray,
    N: np.ndarray,
    y: np.ndarray,
    inner_folds: int,
    grid: List[SupCombo],
    device: str,
    max_epochs: int,
    seed: int,
    thr_opt: str,
) -> Tuple[SupCombo, pd.DataFrame]:
    skf = StratifiedKFold(n_splits=inner_folds, shuffle=True, random_state=seed)
    rows = []

    for gi, combo in enumerate(grid, start=1):
        aucs = []
        for fi, (tr, va) in enumerate(skf.split(np.zeros(len(y)), y), start=1):
            print(hline("-"))
            print(f"[INNER] combo {gi}/{len(grid)} | fold {fi}/{inner_folds} | lr={combo.lr:.4f} | freeze={combo.freeze_encoder}")
            print(hline("-"))

            model, auc, _thr = train_supervised(
                encoder_state=encoder_state,
                cardinalities=cardinalities,
                num_feature_count=num_feature_count,
                numeric_dim=numeric_dim,
                X_tr=X[tr], N_tr=N[tr], y_tr=y[tr],
                X_va=X[va], N_va=N[va], y_va=y[va],
                lr=combo.lr, freeze_encoder=combo.freeze_encoder,
                device=device,
                max_epochs=max_epochs,
                patience=3,
                batch_size=256,
                seed=seed + gi * 100 + fi * 10,
                prefix=f"[IN combo{gi} fold{fi}]",
                thr_opt=thr_opt,
            )
            aucs.append(auc)

        rows.append({
            "combo_id": gi,
            "lr": float(combo.lr),
            "freeze_encoder": bool(combo.freeze_encoder),
            "inner_mean_auc": safe_nanmean(aucs),
            "inner_auc_list": str([float(x) for x in aucs]),
        })

    df = pd.DataFrame(rows).sort_values("inner_mean_auc", ascending=False).reset_index(drop=True)
    best_row = df.iloc[0]
    best_combo = SupCombo(lr=float(best_row["lr"]), freeze_encoder=bool(best_row["freeze_encoder"]))
    return best_combo, df

def run_outer_nested_cv(
    encoder_state: Dict[str, torch.Tensor],
    cardinalities: List[int],
    num_feature_count: int,
    numeric_dim: int,
    X: np.ndarray,
    N: np.ndarray,
    y: np.ndarray,
    outer_folds: int,
    inner_folds: int,
    grid: List[SupCombo],
    device: str,
    inner_max_epochs: int,
    outer_max_epochs: int,
    seed: int,
    thr_opt: str,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    outer = StratifiedKFold(n_splits=outer_folds, shuffle=True, random_state=seed)
    fold_rows = []
    chosen_rows = []

    for ofi, (tr_idx, va_idx) in enumerate(outer.split(np.zeros(len(y)), y), start=1):
        print(hline("="))
        print(f"OUTER FOLD {ofi}/{outer_folds} | train={len(tr_idx)} | val={len(va_idx)}")
        print(hline("="))

        Xtr, Ntr, ytr = X[tr_idx], N[tr_idx], y[tr_idx]
        Xva, Nva, yva = X[va_idx], N[va_idx], y[va_idx]

        best_combo, inner_table = inner_cv_select_combo(
            encoder_state=encoder_state,
            cardinalities=cardinalities,
            num_feature_count=num_feature_count,
            numeric_dim=numeric_dim,
            X=Xtr, N=Ntr, y=ytr,
            inner_folds=inner_folds,
            grid=grid,
            device=device,
            max_epochs=inner_max_epochs,
            seed=seed + ofi * 1000,
            thr_opt=thr_opt,
        )
        print_table(inner_table, f"OUTER {ofi} — Inner-CV combo ranking (by mean AUC)")

        # retrain on outer-train with a small internal val for early stopping + threshold
        tr2, va2 = train_test_split(
            np.arange(len(ytr)),
            test_size=0.10,
            random_state=seed + ofi,
            stratify=ytr,
        )

        print(hline("-"))
        print(f"[OUTER {ofi}] Retrain with best combo: lr={best_combo.lr:.4f} | freeze={best_combo.freeze_encoder}")
        print(hline("-"))

        model, trainval_auc, best_thr = train_supervised(
            encoder_state=encoder_state,
            cardinalities=cardinalities,
            num_feature_count=num_feature_count,
            numeric_dim=numeric_dim,
            X_tr=Xtr[tr2], N_tr=Ntr[tr2], y_tr=ytr[tr2],
            X_va=Xtr[va2], N_va=Ntr[va2], y_va=ytr[va2],
            lr=best_combo.lr,
            freeze_encoder=best_combo.freeze_encoder,
            device=device,
            max_epochs=outer_max_epochs,
            patience=3,
            batch_size=256,
            seed=seed + ofi * 2000,
            prefix=f"[OUTER{ofi}]",
            thr_opt=thr_opt,
        )

        # evaluate on outer val fold using best_thr
        va_loader = DataLoader(SupDataset(Xva, Nva, yva), batch_size=512, shuffle=False, drop_last=False)
        yhat, phat = eval_model_probs(model, va_loader, device=device)
        m = compute_metrics(yhat, phat, thr=best_thr)
        m.update({
            "outer_fold": ofi,
            "outer_val_n": int(len(yva)),
            "chosen_lr": float(best_combo.lr),
            "chosen_freeze_encoder": bool(best_combo.freeze_encoder),
            "chosen_thr": float(best_thr),
            "trainval_internal_auc": float(trainval_auc),
        })
        fold_rows.append(m)

        chosen_rows.append({
            "outer_fold": ofi,
            "lr": float(best_combo.lr),
            "freeze_encoder": bool(best_combo.freeze_encoder),
            "chosen_thr": float(best_thr),
            "outer_val_acc": float(m["acc"]),
            "outer_val_auc": float(m["roc_auc"]),
        })

        print(hline("-"))
        print(f"[OUTER {ofi}] val_acc={m['acc']:.4f} | val_auc={m['roc_auc']:.4f} | thr={best_thr:.3f}")
        print(hline("-"))

    df_folds = pd.DataFrame(fold_rows).sort_values("outer_fold").reset_index(drop=True)
    df_chosen = pd.DataFrame(chosen_rows).sort_values(["outer_val_acc", "outer_val_auc"], ascending=False).reset_index(drop=True)
    return df_folds, df_chosen

def pick_final_combo_from_outer(df_folds: pd.DataFrame) -> SupCombo:
    if len(df_folds) == 0:
        return SupCombo(lr=0.002, freeze_encoder=False)
    g = (
        df_folds.groupby(["chosen_lr", "chosen_freeze_encoder"])["acc"]
        .mean()
        .reset_index()
        .sort_values("acc", ascending=False)
        .reset_index(drop=True)
    )
    best = g.iloc[0]
    return SupCombo(lr=float(best["chosen_lr"]), freeze_encoder=bool(best["chosen_freeze_encoder"]))


# =========================
# 12) Main pipeline
# =========================
def main():
    set_seed(42)

    # -------------------------
    # Config
    # -------------------------
    TARGET = "loan_status"

    BINS_NUMERIC = 128
    VOCAB_CAT = 256

    N_CLIENTS = 4
    FED_ROUNDS = 5
    SSL_BATCH = 256
    SSL_PMASK = 0.22

    SSL_GRID = [
        SSLCombo(lr=0.001, local_epochs=1),
        SSLCombo(lr=0.001, local_epochs=2),
        SSLCombo(lr=0.002, local_epochs=1),
        SSLCombo(lr=0.002, local_epochs=2),
        SSLCombo(lr=0.003, local_epochs=1),
        SSLCombo(lr=0.003, local_epochs=2),
    ]

    OUTER_FOLDS = 5
    INNER_FOLDS = 3
    SUP_GRID = [
        SupCombo(lr=0.001, freeze_encoder=True),
        SupCombo(lr=0.001, freeze_encoder=False),
        SupCombo(lr=0.002, freeze_encoder=True),
        SupCombo(lr=0.002, freeze_encoder=False),
        SupCombo(lr=0.003, freeze_encoder=True),
        SupCombo(lr=0.003, freeze_encoder=False),
    ]

    INNER_MAX_EPOCHS = 6
    OUTER_MAX_EPOCHS = 10
    FINAL_MAX_EPOCHS = 14

    THRESH_OPT_MODE = "acc"  # optimize ACC
    SAVE_PRED_CSV = False

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # -------------------------
    # Load
    # -------------------------
    preferred = os.environ.get("LOAN_CSV", "loan_data.csv")
    csv_path = find_csv_path(preferred)

    print(hline("="))
    print("DNA ENCODING + DNABRIDGE++ + FEDERATED (4 clients) + NESTED CV + FINAL TEST")
    print(hline("="))
    print(f"✅ Using CSV path: {csv_path}")
    print(f"Device: {device}")

    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]

    if TARGET not in df.columns:
        lowmap = {c.lower(): c for c in df.columns}
        if TARGET.lower() in lowmap:
            TARGET = lowmap[TARGET.lower()]
        else:
            raise ValueError(f"Target '{TARGET}' not found. Columns={list(df.columns)}")

    # Drop obvious ID-like columns
    df, dropped = drop_id_like_cols(df, threshold_unique_ratio=0.98)
    if dropped:
        print(f"🧹 Dropping ID-like columns: {dropped}")

    # Dedup early (pre-split)
    fp = sha256_row_fingerprint(df)
    df = df.loc[~fp.duplicated()].reset_index(drop=True)

    # Robust binary target mapping (NO hard-coded Approved/Rejected)
    y_mapped, keep_mask, y_info = infer_binary_target(df[TARGET])
    df = df.loc[keep_mask].reset_index(drop=True)
    y = y_mapped

    if len(df) == 0:
        raise ValueError(
            "After mapping target to 2 classes, dataset is empty.\n"
            f"Target preview (top10): {y_info.get('raw_unique_preview')}"
        )

    num_cols, cat_cols = infer_numeric_vs_categorical(df, TARGET, numeric_threshold=0.80)

    # raw features
    dfX_raw = df.drop(columns=[TARGET]).copy()

    print(hline("-"))
    print(f"Rows: {len(df)}")
    print(f"Target: {TARGET} | target_mode={y_info['mode']} | negative='{y_info['negative_class']}' -> 0 | positive='{y_info['positive_class']}' -> 1")
    print(f"Target preview top10: {y_info.get('raw_unique_preview')}")
    print(f"Numeric cols ({len(num_cols)}): {num_cols}")
    print(f"Categorical cols ({len(cat_cols)}): {cat_cols}")
    print(f"Class counts: {{0: {int(np.sum(y==0))}, 1: {int(np.sum(y==1))}}}")
    print(hline("-"))

    if len(np.unique(y)) < 2:
        raise ValueError("Target has only 1 class after mapping; cannot train a classifier.")

    # -------------------------
    # Holdout split FIRST
    # -------------------------
    idx = np.arange(len(y))
    idx_trval, idx_test = train_test_split(idx, test_size=0.20, random_state=42, stratify=y)
    dfX_trval = dfX_raw.iloc[idx_trval].reset_index(drop=True)
    dfX_test  = dfX_raw.iloc[idx_test].reset_index(drop=True)
    y_trval = y[idx_trval]
    y_test  = y[idx_test]

    print(hline("="))
    print("HOLDOUT SPLIT")
    print(hline("="))
    print(f"TrainVal={len(y_trval)} | Test={len(y_test)}")

    # -------------------------
    # Fit encoding ONCE on TrainVal
    # -------------------------
    enc_fit = fit_dna_encoder(
        df_train=dfX_trval,
        y_train=y_trval,
        target_col=TARGET,
        num_cols=num_cols,
        cat_cols=cat_cols,
        bins_numeric=BINS_NUMERIC,
        vocab_cat=VOCAB_CAT,
        seed=42,
    )

    X_trval, N_trval = transform_dna(enc_fit, dfX_trval)
    X_test,  N_test  = transform_dna(enc_fit, dfX_test)

    offsets, token_base = build_offsets(enc_fit.cardinalities)
    total_tokens = token_base + len(enc_fit.cardinalities) + 1

    print(hline("-"))
    print(f"Encoding sizes: numeric_bins={BINS_NUMERIC} | cat_vocab={VOCAB_CAT}")
    print(f"Encoded local tokens: X_trval={X_trval.shape} | N_trval={N_trval.shape}")
    print(f"Global token space: token_base={token_base} | +mask_tokens(F)={len(enc_fit.cardinalities)} | +CLS=1 => total_tokens={total_tokens}")
    print(f"Example k-mer: numeric token 0 => {int_to_kmer(0, enc_fit.k_num)} | cat token 0 => {int_to_kmer(0, enc_fit.k_cat)}")
    print(hline("-"))

    # -------------------------
    # Encoding validation tables (ONCE on TrainVal)
    # -------------------------
    E1 = validate_numeric_dna_quantile(
        dfX=dfX_trval, y=y_trval,
        num_cols=num_cols,
        edges_by_col=enc_fit.edges_by_col,
        bins_numeric=BINS_NUMERIC
    )
    print_table(E1, "TABLE E1 — Numeric DNA (Supervised-Quantile) Encoding Validation (NO acc/auc)")

    E2 = validate_categorical_vocab(
        dfX=dfX_trval,
        cat_cols=cat_cols,
        cat_encoder=enc_fit.cat_encoder,
        vocab_cat=VOCAB_CAT
    )
    print_table(E2, "TABLE E2 — Categorical Vocab Validation (Collision-free; NO acc/auc)")

    E3 = bootstrap_stability_numeric_edges(
        dfX=dfX_trval, y=y_trval,
        num_cols=num_cols,
        edges_by_col=enc_fit.edges_by_col,
        bins_numeric=BINS_NUMERIC,
        B=6
    )
    print_table(E3, "TABLE E3 — Bootstrap Stability of Numeric (Supervised) Edges (NO acc/auc)")

    E4 = token_robustness_under_mutation(
        dfX=dfX_trval,
        num_cols=num_cols, cat_cols=cat_cols,
        edges_by_col=enc_fit.edges_by_col,
        cat_encoder=enc_fit.cat_encoder,
        bins_numeric=BINS_NUMERIC,
        mutation_rate=0.10
    )
    print_table(E4, "TABLE E4 — Token Robustness Under Mutation (NO acc/auc)")

    # -------------------------
    # Build 4 clients from TrainVal
    # -------------------------
    skf_clients = StratifiedKFold(n_splits=N_CLIENTS, shuffle=True, random_state=42)
    client_splits = []
    for _, client_idx in skf_clients.split(np.zeros(len(y_trval)), y_trval):
        client_splits.append(client_idx)

    # -------------------------
    # Init SSL model
    # -------------------------
    encoder_init = DNABridgePPEncoder(
        cardinalities=enc_fit.cardinalities,
        num_feature_count=len(num_cols),
        emb_dim=96, n_layers=2, n_heads=4, ff_dim=192, dropout=0.10
    ).to(device)
    ssl_model_init = DNABridgePP_SSL(encoder_init).to(device)

    base_ssl_state = {k: v.detach().cpu().clone() for k, v in ssl_model_init.state_dict().items()}
    encoder_param_count = encoder_init.count_params()

    print(hline("="))
    print("MODEL SUMMARY (DNABRIDGE++ encoder)")
    print(hline("="))
    print(f"Encoder params: {encoder_param_count}")
    print(f"Client grid combos per client: {len(SSL_GRID)}")
    print(f"Total client combos evaluated: {len(SSL_GRID) * N_CLIENTS}")
    print(f"NestedCV combos per outer fold: {len(SUP_GRID)}")
    print(hline("="))

    # -------------------------
    # Client-wise SSL hyperparam search
    # -------------------------
    client_best = []
    all_client_rows = []

    for ci, cidx in enumerate(client_splits):
        Xc = X_trval[cidx]
        Nc = N_trval[cidx]
        yc = y_trval[cidx]

        tr_i, va_i = train_test_split(np.arange(len(cidx)), test_size=0.20, random_state=42, stratify=yc)

        tr_loader = DataLoader(TokenOnlyDataset(Xc[tr_i], Nc[tr_i]), batch_size=SSL_BATCH, shuffle=True, drop_last=False)
        va_loader = DataLoader(TokenOnlyDataset(Xc[va_i], Nc[va_i]), batch_size=SSL_BATCH, shuffle=False, drop_last=False)

        print(hline("-"))
        print(f"[CLIENT {ci}] Hyperparam search | local_n={len(cidx)} | train={len(tr_i)} | val={len(va_i)} | combos={len(SSL_GRID)}")
        print(hline("-"))

        best_loss = float("inf")
        best_combo = None

        for k, combo in enumerate(SSL_GRID, start=1):
            enc = DNABridgePPEncoder(
                cardinalities=enc_fit.cardinalities,
                num_feature_count=len(num_cols),
                emb_dim=96, n_layers=2, n_heads=4, ff_dim=192, dropout=0.10
            ).to(device)
            sslm = DNABridgePP_SSL(enc).to(device)
            sslm.load_state_dict(base_ssl_state, strict=True)

            opt = torch.optim.AdamW(sslm.parameters(), lr=combo.lr, weight_decay=1e-4)

            for ep in range(combo.local_epochs):
                _ = run_ssl_epoch(
                    model_ssl=sslm,
                    loader=tr_loader,
                    cardinalities=enc_fit.cardinalities,
                    num_feature_count=len(num_cols),
                    device=device, opt=opt, train=True,
                    p_mask=SSL_PMASK,
                    seed=42 + ci * 1000 + ep * 10 + k
                )

            val_loss = run_ssl_epoch(
                model_ssl=sslm,
                loader=va_loader,
                cardinalities=enc_fit.cardinalities,
                num_feature_count=len(num_cols),
                device=device, opt=None, train=False,
                p_mask=SSL_PMASK,
                seed=999 + ci * 1000 + k
            )

            print(f"[CLIENT {ci}] combo {k:02d}/{len(SSL_GRID)} | lr={combo.lr:.3f} | local_epochs={combo.local_epochs} | val_ssl_loss={val_loss:.6f}")

            all_client_rows.append({
                "client": ci,
                "combo_id": k,
                "lr": combo.lr,
                "local_epochs": combo.local_epochs,
                "val_ssl_loss": val_loss,
            })

            if val_loss < best_loss:
                best_loss = val_loss
                best_combo = combo

        print(hline("-"))
        print(f"[CLIENT {ci}] ✅ BEST: lr={best_combo.lr:.3f} | local_epochs={best_combo.local_epochs} | best_val_ssl_loss={best_loss:.6f}")
        print(hline("-"))

        client_best.append({
            "client": ci,
            "n_samples": int(len(cidx)),
            "best_lr": float(best_combo.lr),
            "best_local_epochs": int(best_combo.local_epochs),
            "best_val_ssl_loss": float(best_loss),
        })

        df_client = pd.DataFrame([r for r in all_client_rows if r["client"] == ci]).sort_values("val_ssl_loss", ascending=True)
        print_table(df_client, f"CLIENT {ci} — All SSL hyperparam combinations (ranked)")

    T_client_best = pd.DataFrame(client_best)
    print_table(T_client_best, "TABLE 1 — Client-wise Best SSL Hyperparams")

    # -------------------------
    # Federated SSL training (FedAvg)
    # -------------------------
    print(hline("="))
    print("FEDERATED TRAINING (FedAvg) | DNABRIDGE++ (MaskedRecon + VICReg) | 4 clients")
    print(hline("="))
    print(f"Rounds: {FED_ROUNDS} | Batch size: {SSL_BATCH} | Device: {device}")

    g_tr_i, g_va_i = train_test_split(np.arange(len(y_trval)), test_size=0.10, random_state=42, stratify=y_trval)
    global_ssl_val_loader = DataLoader(TokenOnlyDataset(X_trval[g_va_i], N_trval[g_va_i]), batch_size=SSL_BATCH, shuffle=False, drop_last=False)

    enc_global = DNABridgePPEncoder(
        cardinalities=enc_fit.cardinalities,
        num_feature_count=len(num_cols),
        emb_dim=96, n_layers=2, n_heads=4, ff_dim=192, dropout=0.10
    ).to(device)
    ssl_global = DNABridgePP_SSL(enc_global).to(device)
    ssl_global.load_state_dict(base_ssl_state, strict=True)

    fed_rows = []
    for rd in range(1, FED_ROUNDS + 1):
        print(hline("-"))
        print(f"[ROUND {rd}/{FED_ROUNDS}]")
        print(hline("-"))

        client_states = []
        client_weights = []

        for ci, cidx in enumerate(client_splits):
            Xc = X_trval[cidx]
            Nc = N_trval[cidx]
            loader = DataLoader(TokenOnlyDataset(Xc, Nc), batch_size=SSL_BATCH, shuffle=True, drop_last=False)

            enc_loc = DNABridgePPEncoder(
                cardinalities=enc_fit.cardinalities,
                num_feature_count=len(num_cols),
                emb_dim=96, n_layers=2, n_heads=4, ff_dim=192, dropout=0.10
            ).to(device)
            ssl_loc = DNABridgePP_SSL(enc_loc).to(device)

            ssl_loc.load_state_dict({k: v.detach().cpu() for k, v in ssl_global.state_dict().items()}, strict=True)

            lr = float(T_client_best.loc[T_client_best["client"] == ci, "best_lr"].values[0])
            local_epochs = int(T_client_best.loc[T_client_best["client"] == ci, "best_local_epochs"].values[0])

            opt = torch.optim.AdamW(ssl_loc.parameters(), lr=lr, weight_decay=1e-4)
            for ep in range(local_epochs):
                _ = run_ssl_epoch(
                    model_ssl=ssl_loc,
                    loader=loader,
                    cardinalities=enc_fit.cardinalities,
                    num_feature_count=len(num_cols),
                    device=device, opt=opt, train=True,
                    p_mask=SSL_PMASK,
                    seed=10000 + rd * 100 + ci * 10 + ep
                )

            client_states.append({k: v.detach().cpu().clone() for k, v in ssl_loc.state_dict().items()})
            client_weights.append(int(len(cidx)))

        new_state = fedavg(client_states, client_weights)
        ssl_global.load_state_dict(new_state, strict=True)

        val_loss = run_ssl_epoch(
            model_ssl=ssl_global,
            loader=global_ssl_val_loader,
            cardinalities=enc_fit.cardinalities,
            num_feature_count=len(num_cols),
            device=device, opt=None, train=False,
            p_mask=SSL_PMASK,
            seed=20000 + rd
        )

        print(f"[ROUND {rd}] global_ssl_val_loss={val_loss:.6f}")
        fed_rows.append({"round": rd, "global_ssl_val_loss": float(val_loss)})

    T_fed = pd.DataFrame(fed_rows)
    print_table(T_fed, "TABLE — Federated rounds (global SSL val loss)")

    encoder_state_global = {k: v.detach().cpu().clone() for k, v in ssl_global.encoder.state_dict().items()}

    # -------------------------
    # Nested CV supervised fine-tune (on TrainVal)
    # -------------------------
    print(hline("="))
    print("NESTED CV (Supervised Fine-tuning)")
    print(hline("="))

    df_folds, df_chosen = run_outer_nested_cv(
        encoder_state=encoder_state_global,
        cardinalities=enc_fit.cardinalities,
        num_feature_count=len(num_cols),
        numeric_dim=N_trval.shape[1],
        X=X_trval, N=N_trval, y=y_trval,
        outer_folds=OUTER_FOLDS,
        inner_folds=INNER_FOLDS,
        grid=SUP_GRID,
        device=device,
        inner_max_epochs=INNER_MAX_EPOCHS,
        outer_max_epochs=OUTER_MAX_EPOCHS,
        seed=42,
        thr_opt=THRESH_OPT_MODE,
    )

    print_table(df_folds, "TABLE 2 — Outer-fold validation metrics (with chosen params)")
    print_table(df_chosen, "TABLE — Per-fold chosen params + outer val summary")

    final_combo = pick_final_combo_from_outer(df_folds)
    print(hline("-"))
    print(f"FINAL chosen supervised combo (by mean outer-fold ACC): lr={final_combo.lr:.4f} | freeze={final_combo.freeze_encoder}")
    print(hline("-"))

    # -------------------------
    # Final train on TrainVal, evaluate on Holdout Test
    # -------------------------
    tr2, va2 = train_test_split(
        np.arange(len(y_trval)),
        test_size=0.10,
        random_state=4242,
        stratify=y_trval,
    )

    final_model, final_auc_internal, final_thr = train_supervised(
        encoder_state=encoder_state_global,
        cardinalities=enc_fit.cardinalities,
        num_feature_count=len(num_cols),
        numeric_dim=N_trval.shape[1],
        X_tr=X_trval[tr2], N_tr=N_trval[tr2], y_tr=y_trval[tr2],
        X_va=X_trval[va2], N_va=N_trval[va2], y_va=y_trval[va2],
        lr=final_combo.lr,
        freeze_encoder=final_combo.freeze_encoder,
        device=device,
        max_epochs=FINAL_MAX_EPOCHS,
        patience=4,
        batch_size=256,
        seed=9001,
        prefix="[FINAL]",
        thr_opt=THRESH_OPT_MODE,
    )

    test_loader = DataLoader(SupDataset(X_test, N_test, y_test), batch_size=512, shuffle=False, drop_last=False)
    yhat_test, phat_test = eval_model_probs(final_model, test_loader, device=device)
    test_metrics = compute_metrics(yhat_test, phat_test, thr=final_thr)

    T_test = pd.DataFrame([{
        **test_metrics,
        "chosen_thr": float(final_thr),
        "internal_val_auc": float(final_auc_internal),
        "final_lr": float(final_combo.lr),
        "final_freeze_encoder": bool(final_combo.freeze_encoder),
        "test_n": int(len(y_test)),
    }])
    print_table(T_test, "TABLE 3 — Final Holdout Test Metrics")

    # sample predictions
    sample_n = min(20, len(y_test))
    pred_df = pd.DataFrame({
        "y_true": yhat_test[:sample_n].astype(int),
        "prob_pos": phat_test[:sample_n].astype(float),
        "y_pred": (phat_test[:sample_n] >= final_thr).astype(int),
    })
    print_table(pred_df, "TABLE — Sample predictions (first 20)")

    if SAVE_PRED_CSV:
        out_csv = "dnabridgepp_predictions.csv"
        pd.DataFrame({"y_true": yhat_test.astype(int), "prob_pos": phat_test.astype(float)}).to_csv(out_csv, index=False)
        print(f"Saved predictions to {out_csv}")

    # -------------------------
    # Totals / configs summary
    # -------------------------
    totals = pd.DataFrame([{
        "n_rows_total_after_clean": int(len(df)),
        "n_trainval": int(len(y_trval)),
        "n_test": int(len(y_test)),
        "n_features_total": int(len(enc_fit.cardinalities)),
        "n_numeric_features": int(len(num_cols)),
        "n_categorical_features": int(len(cat_cols)),
        "encoder_params": int(encoder_param_count),
        "ssl_grid_per_client": int(len(SSL_GRID)),
        "n_clients": int(N_CLIENTS),
        "ssl_total_client_configs": int(len(SSL_GRID) * N_CLIENTS),
        "fed_rounds": int(FED_ROUNDS),
        "sup_grid_size": int(len(SUP_GRID)),
        "outer_folds": int(OUTER_FOLDS),
        "inner_folds": int(INNER_FOLDS),
        "nested_total_trainings_approx": int(OUTER_FOLDS * INNER_FOLDS * len(SUP_GRID)),
        "final_threshold": float(final_thr),
    }])
    print_table(totals, "TABLE 4 — Totals / counts / final params")


if __name__ == "__main__":
    main()

DNA ENCODING + DNABRIDGE++ + FEDERATED (4 clients) + NESTED CV + FINAL TEST
✅ Using CSV path: /kaggle/input/loan-approval-classification-data/loan_data.csv
Device: cuda
------------------------------------------------------------------------------------------------------------------------
Rows: 45000
Target: loan_status | target_mode=numeric | negative='0' -> 0 | positive='1' -> 1
Target preview top10: {'0': 35000, '1': 10000}
Numeric cols (8): ['person_age', 'person_income', 'person_emp_exp', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'credit_score']
Categorical cols (5): ['person_gender', 'person_education', 'person_home_ownership', 'loan_intent', 'previous_loan_defaults_on_file']
Class counts: {0: 35000, 1: 10000}
------------------------------------------------------------------------------------------------------------------------
HOLDOUT SPLIT
TrainVal=36000 | Test=9000
-----------------------------------------------------------------------

| feature                    |   bins_vocab_size |   edges_count | edges_preview_first10                                                                                                     |   unique_tokens_used |   silhouette |   davies_bouldin |   calinski_harabasz |   quantization_mse |   token_entropy |   effective_tokens_expH |   mutual_info_with_target |   normalized_mi |
|----------------------------|-------------------|---------------|---------------------------------------------------------------------------------------------------------------------------|----------------------|--------------|------------------|---------------------|--------------------|-----------------|-------------------------|---------------------------|-----------------|
| person_age                 |               128 |            21 | [21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0]...                                                           |                   22 |     0.959747 |         0.316418 |         4123.036220 |           1.605870 |        2.807846 |               16.574177 |                  0.000941 |        0.001777 |
| person_income              |               128 |           125 | [16411.25, 19349.296875, 19958.1875, 22862.15625, 24160.8125, 25112.625, 27260.625, 29720.90625, 29743.96875, 29875.0]... |                  125 |     0.233317 |         0.707226 |           22.304218 |  3792212705.810807 |        4.629535 |              102.466383 |                  0.071048 |        0.134128 |
| person_emp_exp             |               128 |            20 | [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0]...                                                                     |                   20 |     0.963999 |         0.314183 |         4410.597050 |           1.654721 |        2.660043 |               14.296903 |                  0.000651 |        0.001230 |
| loan_amnt                  |               128 |            67 | [1325.0, 1637.75, 1700.0, 1825.0, 2000.0, 2291.875, 2400.0, 2500.0, 3000.0, 3113.75]...                                   |                   68 |     0.564683 |         0.579596 |         2652.571628 |      499688.175923 |        3.687069 |               39.927659 |                  0.011081 |        0.020918 |
| loan_int_rate              |               128 |           107 | [5.79, 5.99, 6.17, 6.25, 6.56, 6.76, 6.98, 6.99, 7.11, 7.14]...                                                           |                  108 |     0.452594 |         0.564497 |        30880.582066 |           0.006212 |        4.345154 |               77.103932 |                  0.067416 |        0.127271 |
| loan_percent_income        |               128 |            35 | [0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.11]...                                                            |                   36 |     0.971203 |         0.186603 |        25362.376311 |           0.000023 |        3.391891 |               29.722094 |                  0.081097 |        0.153099 |
| cb_person_cred_hist_length |               128 |            15 | [2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0]...                                                                   |                   15 |     0.971824 |         0.406106 |         7959.724179 |           0.289504 |        2.362573 |               10.618232 |                  0.000293 |        0.000554 |
| credit_score               |               128 |            84 | [507.0, 525.0, 526.0, 527.0, 537.0, 546.0, 554.0, 561.0, 566.0, 571.0]...                                                 |                   84 |     0.622880 |         0.417311 |         7311.804390 |           8.955966 |        4.351635 |               77.605230 |                  0.001211 |        0.002285 |


TABLE E2 — Categorical Vocab Validation (Collision-free; NO acc/auc)


| feature                        |   vocab_size |   n_samples |   unique_raw_categories |   occupied_tokens |   oov_rate |   token_entropy |   effective_tokens_expH |   top_token_share |   mean_token_load |   max_token_load |
|--------------------------------|--------------|-------------|-------------------------|-------------------|------------|-----------------|-------------------------|-------------------|-------------------|------------------|
| person_gender                  |          256 |       36000 |                       2 |                 2 |   0.000000 |        0.688217 |                1.990163 |          0.549611 |      18000.000000 |            19786 |
| person_education               |          256 |       36000 |                       5 |                 5 |   0.000000 |        1.414762 |                4.115508 |          0.298278 |       7200.000000 |            10738 |
| person_home_ownership          |          256 |       36000 |                       4 |                 4 |   0.000000 |        0.900318 |                2.460385 |          0.520278 |       9000.000000 |            18730 |
| loan_intent                    |          256 |       36000 |                       6 |                 6 |   0.000000 |        1.773587 |                5.891950 |          0.202667 |       6000.000000 |             7296 |
| previous_loan_defaults_on_file |          256 |       36000 |                       2 |                 2 |   0.000000 |        0.693040 |                1.999787 |          0.507306 |      18000.000000 |            18263 |


TABLE E3 — Bootstrap Stability of Numeric (Supervised) Edges (NO acc/auc)


| feature                    |   bins_vocab_size |   base_edges_count |   base_std_scale |   bootstrap_B |   mean_abs_edge_deviation |   std_abs_edge_deviation |   mean_token_agreement_vs_base |   std_token_agreement_vs_base |   mean_bins_used_fixed_subset |
|----------------------------|-------------------|--------------------|------------------|---------------|---------------------------|--------------------------|--------------------------------|-------------------------------|-------------------------------|
| person_age                 |               128 |                 21 |         6.045270 |             6 |                  0.555608 |                 0.785782 |                       0.991833 |                      0.011564 |                     22.333333 |
| person_income              |               128 |                125 |     84315.801379 |             6 |                371.328917 |               107.041466 |                       0.703333 |                      0.047359 |                    122.166667 |
| person_emp_exp             |               128 |                 20 |         6.064155 |             6 |                  0.610034 |                 0.835398 |                       0.981833 |                      0.021507 |                     20.333333 |
| loan_amnt                  |               128 |                 67 |      6301.359905 |             6 |               5494.584587 |              3770.018197 |                       0.228083 |                      0.107110 |                     62.833333 |
| loan_int_rate              |               128 |                107 |         2.976247 |             6 |                  3.122424 |                 1.639322 |                       0.148750 |                      0.044644 |                    101.500000 |
| loan_percent_income        |               128 |                 35 |         0.086947 |             6 |                  0.008516 |                 0.011784 |                       0.968417 |                      0.054330 |                     36.166667 |
| cb_person_cred_hist_length |               128 |                 15 |         3.878655 |             6 |                  1.466865 |                 1.123825 |                       0.826500 |                      0.235800 |                     15.500000 |
| credit_score               |               128 |                 84 |        50.291647 |             6 |                105.860449 |                68.257543 |                       0.085000 |                      0.035979 |                     90.666667 |

... (30 outputs left out)


TABLE — Sample predictions (first 20)


|   y_true |   prob_pos |   y_pred |
|----------|------------|----------|
| 0.000000 |   0.006414 | 0.000000 |
| 0.000000 |   0.000003 | 0.000000 |
| 0.000000 |   0.000000 | 0.000000 |
| 0.000000 |   0.000000 | 0.000000 |
| 1.000000 |   0.955361 | 1.000000 |
| 1.000000 |   0.586409 | 0.000000 |
| 0.000000 |   0.055527 | 0.000000 |
| 0.000000 |   0.000000 | 0.000000 |
| 0.000000 |   0.531694 | 0.000000 |
| 0.000000 |   0.012419 | 0.000000 |
| 0.000000 |   0.000003 | 0.000000 |
| 0.000000 |   0.000001 | 0.000000 |
| 0.000000 |   0.049909 | 0.000000 |
| 0.000000 |   0.564031 | 0.000000 |
| 0.000000 |   0.801413 | 0.000000 |
| 0.000000 |   0.278399 | 0.000000 |
| 0.000000 |   0.000001 | 0.000000 |
| 0.000000 |   0.000008 | 0.000000 |
| 0.000000 |   0.198543 | 0.000000 |
| 0.000000 |   0.000013 | 0.000000 |


TABLE 4 — Totals / counts / final params


|   n_rows_total_after_clean |   n_trainval |      n_test |   n_features_total |   n_numeric_features |   n_categorical_features |   encoder_params |   ssl_grid_per_client |   n_clients |   ssl_total_client_configs |   fed_rounds |   sup_grid_size |   outer_folds |   inner_folds |   nested_total_trainings_approx |   final_threshold |
|----------------------------|--------------|-------------|--------------------|----------------------|--------------------------|------------------|-----------------------|-------------|----------------------------|--------------|-----------------|---------------|---------------|---------------------------------|-------------------|
|               45000.000000 | 36000.000000 | 9000.000000 |          13.000000 |             8.000000 |                 5.000000 |    393408.000000 |              6.000000 |    4.000000 |                  24.000000 |     5.000000 |        6.000000 |      5.000000 |      3.000000 |                       90.000000 |          0.870000 |